In [2]:
import pandas as pd
import numpy as np

print("Environment working")

Environment working


In [3]:
import pandas as pd
import numpy as np

# File paths
review_path = "../data/raw/yelp_academic_dataset_review.json"
business_path = "../data/raw/yelp_academic_dataset_business.json"
user_path = "../data/raw/yelp_academic_dataset_user.json"
print("Paths loaded successfully")

Paths loaded successfully


In [4]:
# STEP 3
# Loading dataframes
# For initial exploration, we will load a sample of the reviews and users datasets to avoid memory issues, while loading the entire business dataset to analyze restaurant-related information.
# This approach allows us to get a sense of the structure and content of the datasets without overwhelming our system's memory, while still providing us with enough data to perform meaningful analysis on restaurant reviews and business information.

reviews_sample = pd.read_json(
    review_path,
    lines=True,
    nrows=5000
)

business_df = pd.read_json(
    business_path,
    lines=True,
)

users_sample = pd.read_json(
    user_path,
    lines=True,
    nrows=5000
)

print("Reviews shape:", reviews_sample.shape)
print("Business dataset shape:")
print(business_df.shape)
print("Users shape:", users_sample.shape)

Reviews shape: (5000, 9)
Business dataset shape:
(150346, 14)
Users shape: (5000, 22)


In [5]:
# STEP 4
# Inspecting columns of each dataset
# This will help us understand the structure of the data and identify key features for analysis.

print("REVIEWS COLUMNS")
print(reviews_sample.columns)

print("\nBUSINESS COLUMNS")
print(business_df.columns)

print("\nUSER COLUMNS")
print(users_sample.columns)

REVIEWS COLUMNS
Index(['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny',
       'cool', 'text', 'date'],
      dtype='str')

BUSINESS COLUMNS
Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='str')

USER COLUMNS
Index(['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny',
       'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos'],
      dtype='str')


In [6]:
# STEP 5
# Displaying first few rows of each dataset to get a sense of the data
reviews_sample.head(2)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18


In Reviews, there are important fields like: user_id, business_id, stars, text, date

These serves as our behavioral signal layer.

In [7]:
# STEP 6
# Displaying first few rows of business dataset

business_df.head(2)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."


In Businesses

Fields like: categories,city, attributes,stars

This serves as our contextual environment layer.

In [8]:

# STEP 7
# Displaying first few rows of users dataset

users_sample.head(2)

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946


In Users fields like: review_count, average_stars, fans

These serves as our behavioral metadata layer.

STEP 8 
Filtering Restaurant Businesses

We only want businesses related to: restaurants, food, cafes, bars because this domain contains rich emotional/social behavior.

In [9]:
# Keep only businesses with restaurant-related categories
# This will help us focus our analysis on the restaurant industry, which is a major part of Yelp's business and user interactions.
# filters businesses whose categories contain: Restaurant, Food, Coffee, Cafe, Bar

restaurant_businesses = business_df[
    business_df["categories"]
    .fillna("")
    .str.contains(
        "Restaurant|Food|Coffee|Cafe|Bar",
        case=False,
        regex=True
    )
]

print("Restaurant businesses:")
print(restaurant_businesses.shape)

Restaurant businesses:
(68696, 14)


In [10]:
# STEP 9
# Inspect Restaurant businesses

restaurant_businesses[
    ["business_id", "name", "categories", "city"]
].head(10)

,business_id,name,categories,city
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",Philadelphia
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,"Brewpubs, Breweries, Food",Green Lane
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",Ashland City
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,"Pubs, Restaurants, Italian, Bars, American (Tr...",Affton
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",Nashville
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,"Vietnamese, Food, Restaurants, Food Trucks",Tampa Bay
12,il_Ro8jwPlHresjw9EGmBg,Denny's,"American (Traditional), Restaurants, Diners, B...",Indianapolis
14,0bPLkL0QhhPO5kt1_EXmNQ,Zio's Italian Market,"Food, Delis, Italian, Bakeries, Restaurants",Largo
15,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,"Sushi Bars, Restaurants, Japanese",Philadelphia
19,ROeacJQwBeh05Rqg7F6TCg,BAP,"Korean, Restaurants",Philadelphia


In [11]:
# STEP 10
# Extract unique restaurant business IDs

restaurant_ids = set(
    restaurant_businesses["business_id"]
)

print("Number of restaurant IDs:")
print(len(restaurant_ids))

Number of restaurant IDs:
68696


In [12]:
# STEP 11
# Filter reviews to include only those related to the restaurant businesses identified above. 
# This will allow us to analyze user feedback specifically for restaurants, which is crucial for understanding customer satisfaction and business performance in this sector.
# This makes personas cleaner, preferences clearer, recommendations better

restaurant_reviews = reviews_sample[
    reviews_sample["business_id"].isin(
        restaurant_businesses["business_id"]
    )
]

print("Restaurant reviews shape:")
print(restaurant_reviews.shape)

Restaurant reviews shape:
(3987, 9)


In [13]:
# STEP 12
#  Inspect restaurant reviews
# This will give us insights into the types of feedback customers are providing for restaurants, which can inform our analysis of customer satisfaction and business performance in this sector.

restaurant_reviews[
    ["user_id", "business_id", "stars", "text"]
].head(5)

,user_id,business_id,stars,text
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,"If you decide to eat here, just be aware it is..."
2,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,Family diner. Had the buffet. Eclectic assortm...
3,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,"Wow! Yummy, different, delicious. Our favo..."
4,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,Cute interior and owner (?) gave us tour of up...
5,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,I am a long term frequent customer of this est...


Finally, I am looking at real behavioral data. I can now see:

emotions
complaints
enthusiasm
sarcasm
value judgments

This is a raw psychological signal.

STEP 13: Find Sweet-Spot Users

Now we identify behaviorally rich users.

In [14]:
# Count reviews per user
# This will help us identify active users and understand the distribution of reviews among users, which can inform our analysis of user behavior and preferences in the restaurant sector.

user_review_counts = (
    restaurant_reviews
    .groupby("user_id")
    .size()
    .reset_index(name="review_count")
)

# Keep users with 15–50 reviews

sweet_spot_users = user_review_counts[
    (user_review_counts["review_count"] >= 15) &
    (user_review_counts["review_count"] <= 50)
]

print("Sweet-spot users:")
print(sweet_spot_users.shape)

Sweet-spot users:
(0, 2)


In [15]:
# STEP 13B
# Inspect sweet-spot users
sweet_spot_users.head(10)

,user_id,review_count


In [16]:
# STEP 14
# Read review data in chunks
# This approach allows us to process large datasets without running into memory issues, enabling us to filter and analyze reviews related to restaurants efficiently.
# By reading the review data in chunks, we can handle the large size of the dataset while still extracting relevant information for our analysis of restaurant reviews.

chunk_size = 100000

restaurant_review_chunks = []

for chunk in pd.read_json(
    review_path,
    lines=True,
    chunksize=chunk_size
):
    
    filtered_chunk = chunk[
        chunk["business_id"].isin(restaurant_ids)
    ]
    
    restaurant_review_chunks.append(filtered_chunk)

    print(
        f"Processed chunk with "
        f"{len(filtered_chunk)} restaurant reviews"
    )

Processed chunk with 80537 restaurant reviews
Processed chunk with 80648 restaurant reviews
Processed chunk with 79453 restaurant reviews
Processed chunk with 75590 restaurant reviews
Processed chunk with 71692 restaurant reviews
Processed chunk with 70100 restaurant reviews
Processed chunk with 68485 restaurant reviews
Processed chunk with 79838 restaurant reviews
Processed chunk with 81158 restaurant reviews
Processed chunk with 80883 restaurant reviews
Processed chunk with 77279 restaurant reviews
Processed chunk with 72916 restaurant reviews
Processed chunk with 70352 restaurant reviews
Processed chunk with 68087 restaurant reviews
Processed chunk with 78749 restaurant reviews
Processed chunk with 80778 restaurant reviews
Processed chunk with 80846 restaurant reviews
Processed chunk with 77760 restaurant reviews
Processed chunk with 73650 restaurant reviews
Processed chunk with 69105 restaurant reviews
Processed chunk with 69018 restaurant reviews
Processed chunk with 77708 restaur

In [17]:
# Step 15
# Concatenate all filtered chunks into a single DataFrame
# This will give us a complete dataset of restaurant reviews that we can use for further analysis, 
# such as sentiment analysis, user behavior analysis, and business performance evaluation in the restaurant sector.
# This is our real behavioral corpus.

restaurant_reviews = pd.concat(
    restaurant_review_chunks,
    ignore_index=True
)

print("Final restaurant reviews shape:")
print(restaurant_reviews.shape)

Final restaurant reviews shape:
(5259993, 9)


In [18]:
# STEP 16
# Count reviews per user
# This will help us identify users who have reviewed a moderate number of restaurants, which might be indicative of engaged reviewers.

user_review_counts = (
    restaurant_reviews
    .groupby("user_id")
    .size()
    .reset_index(name="review_count")
)

sweet_spot_users = user_review_counts[
    (user_review_counts["review_count"] >= 15) &
    (user_review_counts["review_count"] <= 50)
]

print("Sweet-spot users:")
print(sweet_spot_users.shape)

Sweet-spot users:
(40884, 2)


In [19]:
# STEP 17
# Curate a sample of sweet-spot users for further analysis, 
# ensuring we have a manageable number of users to work with while still capturing a representative subset of engaged reviewers.

curated_users = sweet_spot_users.sample(
    n=300,
    random_state=42
)

print(curated_users.shape)

(300, 2)


In [20]:
# Step 18
# Filter restaurant reviews to include only those from the curated sweet-spot users.
# This will allow us to focus our analysis on a specific subset of engaged users, which can provide more meaningful insights into user behavior and preferences in the restaurant sector.

curated_user_ids = set(
    curated_users["user_id"]
)

curated_reviews = restaurant_reviews[
    restaurant_reviews["user_id"]
    .isin(curated_user_ids)
]

print("Curated reviews shape:")
print(curated_reviews.shape)

Curated reviews shape:
(7413, 9)


This reveals a curated corpus of: psychologically rich users, restaurant behaviors, emotional language, preferences, review styles

Enough to build: personas, review simulation, recommendations, conversational reasoning

In [21]:
# STEP 19
# Transforming raw reviews into interpretable human behavioral traits.
# First, we organize and group reviews by user and business, then we can analyze patterns in the review text, star ratings, and other features to derive insights about user preferences, sentiment, and engagement with restaurants. This will help us understand the underlying behaviors and traits of users in the context of restaurant reviews.



user_histories = (
    curated_reviews
    .groupby("user_id")
    .agg({
        "text": list,
        "stars": list,
        "business_id": list,
        "date": list
    })
    .reset_index()
)

print("User histories shape:")
print(user_histories.shape)

User histories shape:
(300, 5)


In [22]:
# STEP 20
# Inspecting a sample user history to understand the structure of the data and the type of information we have for each user, which will help us in deriving behavioral traits and insights from their reviews.
# Change the index to inspect different users and their review histories.

sample_user = user_histories.iloc[70]

print("USER ID:")
print(sample_user["user_id"])

print("\nSTAR RATINGS:")
print(sample_user["stars"][:5])

print("\nFIRST REVIEW:")
print(sample_user["text"][0][:500])

USER ID:
Gcxm0XlnMIW0sUwiYgo4dA

STAR RATINGS:
[4, 5, 5, 4, 5]

FIRST REVIEW:
Went here for Valentine's day. Very crowded but our reservation was on time and server was good. Grilled seafood app was the highlight of the meal. Grilled scallops, calamari, shrimp and cherry tomatoes over arugula perfectly dressed. 5 stars.
Had filet and pescatore for entree. Filet was good but not amazing- 3 stars. Pescatore was delicious- 5 stars. Lobster slightly dry, but  calamari and scallops and garlic white wine sauce were more than enough.
Dessert was a hit and a miss. Chocolate souff


By changing the index, i notice behavioral traces of a real human.

Things i notice:

tone
complaints
enthusiasm
emotional intensity
writing style
priorities

This is where personas emerge.

In [23]:
# STEP 21 — Creating First Persona Features
# Here we are calculating basic features for each user based on their reviews, such as average rating, rating variance, review count, and average review length. 
# These features can help us understand user behavior and preferences in the context of restaurant reviews, which can be useful for building user personas and improving recommendation systems.

persona_features = (
    curated_reviews
    .groupby("user_id")
    .agg(
        avg_rating=("stars", "mean"),
        rating_variance=("stars", "std"),
        review_count=("stars", "count"),
        avg_review_length=(
            "text",
            lambda x: np.mean(
                x.str.len()
            )
        )
    )
    .reset_index()
)

print(persona_features.shape)

persona_features.head()

(300, 5)


,user_id,avg_rating,rating_variance,review_count,avg_review_length
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455


WHAT THESE FEATURES MEAN
  Feature	               Psychological Meaning
- avg_rating	           harsh vs lenient
- rating_variance	       emotional consistency
- review_count	           engagement
- avg_review_length	       verbosity/detail orientation

These are behavioral signals.

In [24]:
# STEP 22 — Adding Sentiment Features
# Now we estimate emotional tone.

In [25]:
from textblob import TextBlob
from tqdm import tqdm

tqdm.pandas()

In [26]:
# Sentiment per review

curated_reviews["sentiment"] = (
    curated_reviews["text"]
    .progress_apply(
        lambda x: TextBlob(x).sentiment.polarity
    )
)

100%|██████████| 7413/7413 [00:14<00:00, 514.96it/s] 


In [27]:
# STEP 23
# Aggregating User Sentiment

sentiment_features = (
    curated_reviews
    .groupby("user_id")
    .agg(
        avg_sentiment=("sentiment", "mean"),
        sentiment_variance=("sentiment", "std")
    )
    .reset_index()
)

sentiment_features.head()

,user_id,avg_sentiment,sentiment_variance
0,-EX1hrPRBqNkVavtMllTCA,0.138507,0.222461
1,-M7fUg7FrdGctKr5f_eMUQ,0.215294,0.162739
2,-WM58wLjtlHlR91xVfM1FQ,0.298114,0.206382
3,-qTtg1D3RidRa4cTB-ftwg,0.425713,0.245910
4,02H49g16MdRoZKoX6IEoFA,0.255776,0.355440


WHAT THESE MEAN
Feature	              Meaning
avg_sentiment	      positivity/negativity
sentiment_variance	  emotional stability

In [28]:
# STEP 24
# Merge Persona Features

persona_df = persona_features.merge(
    sentiment_features,
    on="user_id"
)

print(persona_df.shape)

persona_df.head()

(300, 7)


,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440


In [29]:
# STEP 25 — Interpret Personas

persona_df.describe()

,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,3.884222,1.081557,24.710000,592.957436,0.256042,0.186791
std,0.604832,0.363819,9.462082,367.794262,0.087883,0.067733
min,1.591837,0.000000,15.000000,175.900000,-0.071841,0.066279
25%,3.523810,0.834853,17.000000,346.679412,0.198519,0.138404
50%,3.942810,1.095445,22.000000,499.306548,0.255502,0.172593
75%,4.315789,1.335010,31.000000,751.808527,0.314091,0.225523
max,5.000000,1.999557,50.000000,2534.300000,0.510412,0.430499


This uncovers human archetypes.

Examples:

angry critics
generous reviewers
emotional storytellers
concise pragmatists

In [30]:
# Harsh Users
# These users tend to give lower ratings on average, which may indicate a more critical perspective or higher standards when it comes to restaurant experiences.

persona_df.sort_values(
    by="avg_rating"
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
20,3GKk2POn0VFznH_KuRX8UA,1.591837,1.153227,49,452.938776,0.034711,0.254318
7,0Tsu6-uhw_w9Z3W0Xlnazg,1.687500,1.138347,16,411.125000,0.093389,0.225517
199,gVvkw-bW7hcrs9P5xfwOhw,1.708333,1.366658,24,502.041667,0.031524,0.212126
179,cb5omh0nibWUYN2rws5rdg,1.850000,1.182103,20,529.450000,-0.071841,0.202567
225,mD-IgInk0o8pXrJI-P8pYA,2.266667,1.099784,15,297.333333,0.124339,0.143346


In [31]:
# Lenient Users
# These users tend to give higher ratings on average, which may indicate a more positive outlook or a tendency to be more forgiving in their reviews.

persona_df.sort_values(
    by="avg_rating",
    ascending=False
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
212,jYOK8bu9lIxkVoKvwPC7ig,5.000000,0.000000,31,361.774194,0.382680,0.176973
81,I6x-ZBHeCNlMnmtfUVf5lg,5.000000,0.000000,24,176.208333,0.510412,0.158668
167,_m0Sxb2_kUFtic4wGwIGzw,4.960000,0.200000,25,249.040000,0.485915,0.160752
5,0650daOKAuufqymyOBe3cA,4.937500,0.250000,16,255.937500,0.478739,0.196268
23,4Z2lfaP3d3oOKmEfmc9PCw,4.933333,0.258199,15,828.200000,0.315061,0.154081


In [32]:
# Verbose Users
# These users tend to write longer reviews, which may indicate a higher level of engagement or a desire to provide more detailed feedback.

persona_df.sort_values(
    by="avg_review_length",
    ascending=False
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
168,_p7aWe_YiAZW2m6RnmAMVA,3.325000,1.071484,40,2534.300000,0.121472,0.099103
247,q4Oz1c_OjGu_a8ZkP53Vnw,2.723404,1.378438,47,2529.531915,0.118203,0.093142
54,Do8oscF3LjCl-_pXrmgLXA,4.041667,0.954585,24,2228.416667,0.178727,0.066896
19,2xNYCJxZOrhVsrVMz8mlDQ,3.911765,0.933149,34,2181.441176,0.228747,0.066279
38,9VdyNBdQbaZTrFDmD49e7A,3.736842,1.097578,19,2119.368421,0.181577,0.117300


With these, we can:

characterize users
compare personalities
simulate tendencies
reason about preferences

Next, We will transform numeric persona signals into recognizable human archetypes.

Examples:

harsh critic
emotional foodie
soft-life explorer
concise pragmatist
luxury seeker

This becomes our Dynamic Cognitive Persona Layer.

In [33]:
# STEP 26 — Prepare for Clustering
# Here we are selecting the relevant features for clustering and filling any missing values with 0. 
# This will allow us to group users into distinct personas based on their review behavior and sentiment, which can be useful for targeted marketing, personalized recommendations, and understanding customer segments in the restaurant industry.

clustering_features = persona_df[
    [
        "avg_rating",
        "rating_variance",
        "avg_review_length",
        "avg_sentiment",
        "sentiment_variance"
    ]
].fillna(0)

clustering_features.head()

,avg_rating,rating_variance,avg_review_length,avg_sentiment,sentiment_variance
0,3.250000,1.441725,390.194444,0.138507,0.222461
1,4.090909,1.341963,358.727273,0.215294,0.162739
2,4.041667,1.197068,632.666667,0.298114,0.206382
3,4.733333,0.703732,195.666667,0.425713,0.245910
4,4.272727,1.202451,406.545455,0.255776,0.355440


In [34]:
# STEP 27 — Standardize Features / Scale Features for Clustering
# Standardizing features is crucial for clustering algorithms, especially those that rely on distance metrics (like K-Means), as it ensures that all features contribute equally to the distance calculations.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    clustering_features
)

print(scaled_features.shape)

(300, 5)


In [35]:
# STEP 28 — Create Behavioral Clusters with K-Means
# K-Means is a popular clustering algorithm that partitions data into K distinct clusters based on feature similarity. 
# By applying K-Means to our standardized features, we can identify distinct user personas based on their review behavior and sentiment, which can provide valuable insights for targeted marketing and personalized recommendations in the restaurant industry.

from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5,
    random_state=42
)

persona_df["cluster"] = kmeans.fit_predict(
    scaled_features
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1


WHY THIS MATTERS

The model is now grouping users by:

emotional style
harshness
verbosity
behavioral consistency

This is latent human behavior discovery.

In [36]:
# STEP 29 — Analyze Cluster Distribution. 
# This will help us understand how users are grouped into different personas based on their review behavior and sentiment, 
# which can inform our strategies for targeted marketing, personalized recommendations, and customer segmentation in the restaurant industry.

persona_df["cluster"].value_counts()

cluster
3    101
0     83
1     54
2     46
4     16
Name: count, dtype: int64

In [37]:
# STEP 30 — Summarize Cluster Characteristics
# This will allow us to understand the defining features of each cluster, which can help us interpret the underlying behaviors 
# and traits of users in each persona, providing insights that can inform targeted marketing strategies, personalized recommendations, and customer segmentation in the restaurant industry.
# Understand Each Cluster AND compute average traits per cluster.

cluster_summary = (
    persona_df
    .groupby("cluster")
    [
        [
            "avg_rating", # 
            "avg_review_length", 
            "avg_sentiment",
            "rating_variance"
        ]
    ]
    .mean()
)

cluster_summary

,avg_rating,avg_review_length,avg_sentiment,rating_variance
cluster,,,,
0,4.464400,411.778871,0.340184,0.730411
1,3.749031,345.437584,0.283938,1.425644
2,2.947635,572.800835,0.139540,1.428969
3,3.908610,708.455119,0.235566,1.055194
4,3.869562,1697.071850,0.189596,0.909445


In [38]:
# STEP 31 - Assign Human Archetype Names
# Interpret Clusters with Names


cluster_names = {
    0: "Warm Optimist",
    1: "Reactive Reviewer",
    2: "Harsh Critic",
    3: "Emotional Storyteller",
    4: "Deep Experience Analyst"
}

persona_df["archetype"] = (
    persona_df["cluster"]
    .map(cluster_names)
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer


In [39]:
# STEP 31A - Create Structured Behavioral Descriptors
# 

behavioral_descriptors = {

    0: {
        "archetype": "Warm Optimist",

        "traits": {
            "positivity": "high",
            "verbosity": "moderate",
            "emotional_stability": "stable",
            "review_style": "supportive",
            "expectation_level": "moderate",
            "decision_style": "emotionally positive"
        }
    },

    1: {
        "archetype": "Reactive Reviewer",

        "traits": {
            "positivity": "mixed",
            "verbosity": "moderate",
            "emotional_stability": "volatile",
            "review_style": "emotion-driven",
            "expectation_level": "variable",
            "decision_style": "experience-sensitive"
        }
    },

    2: {
        "archetype": "Harsh Critic",

        "traits": {
            "positivity": "low",
            "verbosity": "high",
            "emotional_stability": "critical",
            "review_style": "analytical",
            "expectation_level": "high",
            "decision_style": "detail-oriented"
        }
    },

    3: {
        "archetype": "Emotional Storyteller",

        "traits": {
            "positivity": "moderate",
            "verbosity": "high",
            "emotional_stability": "reflective",
            "review_style": "narrative",
            "expectation_level": "balanced",
            "decision_style": "emotionally expressive"
        }
    },

    4: {
        "archetype": "Deep Experience Analyst",

        "traits": {
            "positivity": "moderate",
            "verbosity": "very high",
            "emotional_stability": "stable",
            "review_style": "deeply descriptive",
            "expectation_level": "high",
            "decision_style": "deliberative"
        }
    }
}

In [40]:
# STEP 31B — Attach Archetype Names
persona_df["archetype"] = (
    persona_df["cluster"]
    .apply(
        lambda x: behavioral_descriptors[x]["archetype"]
    )
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer


In [41]:
# STEP 31C — Attach Structured Traits
# Add the full trait dictionaries.


persona_df["behavior_profile"] = (
    persona_df["cluster"]
    .apply(
        lambda x: behavioral_descriptors[x]["traits"]
    )
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em..."
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'..."
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'..."
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'..."
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate..."


WHAT THE DATAFRAME NOW CONTAINS

Each user now has:

Column	                Meaning
avg_rating	            rating behavior
avg_review_length	    verbosity
avg_sentiment	        emotional tone
archetype	            human-readable identity
behavior_profile	    machine-readable cognition

In [42]:
# Qualitative Inspection of User Reviews by Archetype
for archetype in persona_df["archetype"].unique():

    print("\n" + "="*80)
    print(f"ARCHETYPE: {archetype}")
    print("="*80)

    sampled_users = (
        persona_df[
            persona_df["archetype"] == archetype
        ]
        .sample(2, random_state=42)   # fewer users
    )

    for _, user_row in sampled_users.iterrows():

        sample_user_id = user_row["user_id"]

        print("\n" + "#"*60)
        print(f"USER ID: {sample_user_id}")
        print("#"*60)

        user_reviews = curated_reviews[
            curated_reviews["user_id"] == sample_user_id
        ]

        for i, (_, row) in enumerate(
            user_reviews.head(1).iterrows(),  # only 1 review
            start=1
        ):

            print("\n" + "-"*50)
            print(f"Review #{i}")
            print(f"Stars: {row['stars']}")
            print("-"*50)

            print(row["text"][:200].replace("\n", " ") + "...")
            print("\n")


ARCHETYPE: Harsh Critic

############################################################
USER ID: oEMXXNNZiYUllytcZLtbvw
############################################################

--------------------------------------------------
Review #1
Stars: 5
--------------------------------------------------
Oh yum! And clean!! Friendly and efficient, too - I've been there four times so far. The third time, my wife wanted green curry, but they'd featured it the day before. We were very disappointed, until...



############################################################
USER ID: Y-NI1hIn1AB4lP3bT-eSog
############################################################

--------------------------------------------------
Review #1
Stars: 5
--------------------------------------------------
I love local businesses.  I don't drink coffee, but I am obsessed with chai tea lattes and Feine makes a great one.  I love that the baristas offer you different milk options too. I've tried both the ...



ARCHETYP

In [43]:
# STEP 32 — Final Persona Summary
persona_df[
    ["user_id", "archetype"]
].head(10)

,user_id,archetype
0,-EX1hrPRBqNkVavtMllTCA,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,Reactive Reviewer
5,0650daOKAuufqymyOBe3cA,Warm Optimist
6,06Yz-YYYa1U9PN37b6UniA,Emotional Storyteller
7,0Tsu6-uhw_w9Z3W0Xlnazg,Harsh Critic
8,0xJwTzZuWOac7ufeTioeig,Emotional Storyteller
9,175DuOm7IPiEy5f1KG9esA,Harsh Critic


In [44]:
# STEP 33- Merge Archetypes Back Into Reviews
curated_reviews = curated_reviews.merge(
    persona_df[
        ["user_id", "archetype"]
    ],
    on="user_id",
    how="left"
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,archetype
0,rGI8UdsEGiGeETFiu1VK1w,WJnyWEe_YK7JO47fcovBVw,hRHhP3fhMy3LktPyQa3s_A,3,0,0,0,Good choice in Union Station St. Louis. I had...,2008-08-20 08:07:02,0.204545,Emotional Storyteller
1,Lun9ta1qn_pmuD1VqxiYmg,9gSuVhKyOx3Qn4oI6EQMaA,NwJoFxmYRDxVGXgPtrjQ3w,4,0,0,0,Went there for lunch and was pleasantly surpri...,2009-10-14 21:16:08,0.106667,Emotional Storyteller
2,Wu99UIXo1jGJeu97KCJzsw,iBQKwkuDvAdTM5gLWHgZwg,8uF-bhJFgT4Tn6DTb27viA,4,0,0,0,I don't think there is anything in this place ...,2017-12-16 01:54:03,0.403333,Warm Optimist
3,bY-J5JBKI9m8fiFm4CwCFA,ZybKys6Kg37xX2LMfvcntg,x4XDkWR9fgP4TItqMr8A8A,5,0,0,0,"Delicious, fresh, and friendly volunteers. Yes...",2014-06-11 16:43:45,0.430556,Reactive Reviewer
4,xHwfbcnzpIKXbFvHG8kRTQ,3kvIOBG06_rikfpk-EHIlQ,bXjnfT69E8DJinX-ifOofA,1,31,5,5,I've never had to write a review based on horr...,2012-11-07 17:45:30,0.131566,Emotional Storyteller


In [57]:
# STEP 34 — Define Value Taxonomy
# This taxonomy will help us categorize and analyze the different aspects of restaurant reviews, 
# allowing us to extract meaningful insights about customer values, preferences, expectations, and experiences. 
# For each review it counts how often value-related language appears.

import re

def extract_value_signals(text, taxonomy):

    text = text.lower()

    scores = {}

    for category, keywords in taxonomy.items():

        score = 0

        for keyword in keywords:

            matches = re.findall(
                rf"\b{re.escape(keyword)}\b",
                text
            )

            score += len(matches)

        scores[category] = score

    return scores

In [67]:
# STEP 34A — Define Value Taxonomy
# Here we are defining a taxonomy of value-related keywords that can be used to analyze reviews.

value_taxonomy = {

    # Economic values
    "affordability": [
        "cheap", "affordable", "budget", "expensive", "overpriced", "pricey",
        "value for money", "manage", "cost", "naira", "save cost", "waste of money",
        "not worth it", "fair price", "discount", "original price", "market price"
    ],


    # Durability & quality expectations
    "durability": [
        "original", "fake", "counterfeit", "rugged", "last long", "strong",
        "weak", "fragile", "repair", "spare parts", "generator", "battery life",
        "heat up", "spoilt", "still working", "tested and trusted", "tokunbo", "new", "used"
    ],


    # Service & staff behaviour (restaurants, repairs, delivery)
    "service_quality": [
        "service", "staff", "waiter", "waitress", "attentive", "rude",
        "friendly", "slow service", "fast response", "customer care",
        "come and fix", "collect and vanish", "follow me around", "attention to detail", "attentive"
        "pushy seller", "polite", "helpful", "unhelpful", "knowledgeable", "incompetent", "courteous", "disrespectful"
    ],


    # Social proof & communal influence
    "social_proof": [
        "neighbour", "friend recommended", "landlord use am", "colleague",
        "family said", "word of mouth", "everybody buying", "popular",
        "trending", "see my neighbour", "trusted by many", "influencer", "celebrity endorsement", "social media hype"
    ],


    # Temporal / efficiency norms
    "time_efficiency": [
        "wait time", "delay", "fast delivery", "slow", "African time",
        "hours", "minutes", "late", "early", "prompt", "wasted my time",
        "traffic", "Lagos traffic", "delivered on time", "arrived late", 
        "arrived early", "on schedule", "behind schedule", "ahead of schedule"
    ],

    # Ambience & atmosphere (restaurants, events)
    "ambience": [
        "ambience", "atmosphere", "decor", "vibes", "music", "aesthetic", "cozy"
        "lighting", "cleanliness", "noise level", "comfortable seating", "romantic", 
        "family-friendly", "decoration choke", "overcrowded", "spacious", "intimate", "loud", "quiet"
    ],


    # Expressiveness & communication style (Pidgin, humour, directness)
    "expressiveness": [
        "abeg", "na wa o", "seems", "sef", "nko", "abi", "o", "ooh",
        "even the ants rejected am", "I spit", "who send you", "na so so",
        "walahi", "mtchew", "chai", "God willing", "not to praise am too much"
        "no be lie", "I swear", "I dey tell you", "I no dey lie", "I no go lie"
        "i go lie?", "It's giving", "food yakpa", "Everywhere first blur"
    ],

    # Product‑specific attributes (food, electronics, fashion, books, etc.)
    "food_quality": [
        "delicious", "bland", "taste", "fresh", "flavor", "authentic", "portion",
        "presentation", "spicy", "sweet", "sour", "fresh", "salty", "umami", "overcooked",
        "undercooked", "stale", "rotten", "mouthwatering", "swallow", "fufu", "eba", 
        "soup", "rice", "portion size", "small",
    ],


    "electronics": [
        "generator", "inverter", "phone", "laptop", "battery", "charger",
        "NEPA", "light", "plug", "heating", "original charger", "waterproof"
        "power bank", "noise cancelling", "wireless", "durable", "fast charging", "long battery life"
    ],


    "fashion": [
        "lace", "ankara", "fabric quality", "zipper", "tailor", "sewn",
        "fit", "size", "colour fast", "shrink", "native wear", "casual wear", 
        "formal wear", "workwear", "party wear", "traditional attire"
    ],


    "books_media": [
        "motivational", "hustle", "inspirational", "educational", "story",
        "grammar", "chapters", "cover", "print quality", "prayer points"
    ],

    # Convenience & accessibility
    "convenience": [
        "fast", "quick", "parking", "location", "accessible", "easy",  "waiting time"
        "near me", "home delivery", "takeaway", "drive-thru", "curbside pickup", "self-service"
    ],

    # Social experience
    "social_experience": [
        "friends", "family", "date", "group", "celebration", "birthday", "hangout"
        "social gathering", "romantic dinner", "family outing", "friend meetup", "special occasion"
        "anniversary", "reunion", "casual hangout", "work event", "holiday celebration"
        "new spot to try", "place to see and be seen", "vibe for socializing", "perfect for groups", "intimate setting"
    ],

    # Aspirational & status‑related cues
    "luxury": [
        "premium", "luxury", "upscale", "fancy", "high-end", "exclusive", "luxurious", "opulent", 
        "lavish", "posh", "sophisticated", "elegant", "glamorous"
    ],

   # Sentiment polarity markers (Nigerian style exaggeration)
    "positive_exaggeration": [
        "the best", "amazing", "perfect", "excellent", "I love am",
        "changed my life", "highly recommend", "must buy" "refreshing", "life-changing", "unforgettable", "top-notch", "five stars", "beyond expectations"
    ],

    "negative_exaggeration": [
        "worst ever", "terrible", "useless", "waste of data", "I spit",
        "never again", "run away", "scam", "fake life"
    ],


    # Cold‑start & exploration signals
    "uncertainty": [
        "not sure", "maybe", "let me test", "first time", "trying",
        "I no know o", "time will tell", "hoping for the best", "heard good things", "heard bad things", "mixed reviews", "on the fence"
    ]

}

In [78]:
sample_review = curated_reviews.iloc[7]["text"]

print(sample_review[:500])

extract_value_signals(
    sample_review,
    value_taxonomy
)

Big fan of this place. They have the best sugar cookies and butter cake. (Although the butter cake starts getting stale within hours so eat it fast) They are always so nice; gave a cookie to my daughter today while we waited in line. Most importantly, I just leaned they make breakfast sandwiches on Fridays, Saturdays and Sundays. They are excellent and come with western potatoes and a croissant. You can see one half of a breakfast sandwich in my picture. All of this for just $6.99. A very good v


{'affordability': 1,
 'durability': 0,
 'service_quality': 0,
 'social_proof': 0,
 'time_efficiency': 1,
 'ambience': 0,
 'expressiveness': 0,
 'food_quality': 1,
 'electronics': 0,
 'fashion': 0,
 'books_media': 0,
 'convenience': 1,
 'social_experience': 0,
 'luxury': 0,
 'positive_exaggeration': 2,
 'negative_exaggeration': 0,
 'uncertainty': 0}

IMPORTANT

This is interpretable behavioral reasoning. It enables us to explain why the system believes something.

In [70]:
# STEP 35 - APPLY VALUE SIGNAL EXTRACTION
# Now we apply this function to all reviews to extract value signals for each review, 
# which will allow us to analyze the presence of different value-related themes in the 
# reviews and understand customer preferences.

from tqdm import tqdm

tqdm.pandas()

curated_reviews["value_signals"] = (
    curated_reviews["text"]
    .progress_apply(
        lambda x: extract_value_signals(
            x,
            value_taxonomy
        )
    )
)

100%|██████████| 7413/7413 [01:00<00:00, 121.66it/s]


In [72]:
# STEP 36 — EXPAND VALUE SIGNALS INTO COLUMNS
# This will allow us to analyze the presence of different value-related themes in the reviews and 
# understand customer preferences in a more structured way, enabling us to identify patterns and 
# insights related to the values expressed in the reviews.

value_df = pd.json_normalize(
    curated_reviews["value_signals"]
)

value_df.head(12)

,affordability,durability,service_quality,social_proof,time_efficiency,ambience,expressiveness,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,2,0,0,0,0,8,0,0,0,0,0,0,0,0,0
4,0,0,3,0,1,0,0,0,0,0,0,0,2,0,0,0,2
5,1,1,1,0,2,0,0,5,0,0,0,0,0,0,1,0,0
6,0,0,1,0,0,1,0,1,0,0,0,0,0,0,1,0,0
7,1,0,0,0,1,0,0,1,0,0,0,1,0,0,2,0,0
8,1,0,0,0,0,0,0,0,0,0,0,1,0,0,3,0,0
9,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


In [ ]:
# STEP 37 — MERGE VALUE FEATURES
# Now each review contains inferred human values.
# This will allow us to analyze the presence of different value-related themes in the reviews 
# and understand customer preferences in a more structured way, enabling us to identify patterns 
# and insights related to the values expressed in the reviews.

curated_reviews = pd.concat(
    [curated_reviews, value_df],
    axis=1
)

curated_reviews.head()



,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,...,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
0,rGI8UdsEGiGeETFiu1VK1w,WJnyWEe_YK7JO47fcovBVw,hRHhP3fhMy3LktPyQa3s_A,3,0,0,0,Good choice in Union Station St. Louis. I had...,2008-08-20 08:07:02,0.204545,...,0,0,0,0,0,0,0,0,0,0
1,Lun9ta1qn_pmuD1VqxiYmg,9gSuVhKyOx3Qn4oI6EQMaA,NwJoFxmYRDxVGXgPtrjQ3w,4,0,0,0,Went there for lunch and was pleasantly surpri...,2009-10-14 21:16:08,0.106667,...,2,0,0,0,0,0,0,0,0,0
2,Wu99UIXo1jGJeu97KCJzsw,iBQKwkuDvAdTM5gLWHgZwg,8uF-bhJFgT4Tn6DTb27viA,4,0,0,0,I don't think there is anything in this place ...,2017-12-16 01:54:03,0.403333,...,0,0,0,0,0,0,0,0,0,0
3,bY-J5JBKI9m8fiFm4CwCFA,ZybKys6Kg37xX2LMfvcntg,x4XDkWR9fgP4TItqMr8A8A,5,0,0,0,"Delicious, fresh, and friendly volunteers. Yes...",2014-06-11 16:43:45,0.430556,...,8,0,0,0,0,0,0,0,0,0
4,xHwfbcnzpIKXbFvHG8kRTQ,3kvIOBG06_rikfpk-EHIlQ,bXjnfT69E8DJinX-ifOofA,1,31,5,5,I've never had to write a review based on horr...,2012-11-07 17:45:30,0.131566,...,0,0,0,0,0,2,0,0,0,2


In [74]:
# STEP 38 — AGGREGATE VALUES PER USER
# Now we infer what users fundamentally care about.
# By aggregating the value signals at the user level, we can identify overarching themes and preferences 
# that characterize each user's reviews, providing deeper insights into their values and priorities when it 
# comes to restaurant experiences.

user_values = (
    curated_reviews
    .groupby("user_id")
    [
        list(value_taxonomy.keys())
    ]
    .mean()
    .reset_index()
)

user_values.head()

,user_id,affordability,durability,service_quality,social_proof,time_efficiency,ambience,expressiveness,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
0,-EX1hrPRBqNkVavtMllTCA,0.166667,0.222222,0.694444,0.000000,0.222222,0.083333,0.083333,0.722222,0.000000,0.000000,0.0,0.250000,0.027778,0.000000,0.166667,0.027778,0.111111
1,-M7fUg7FrdGctKr5f_eMUQ,0.045455,0.136364,0.636364,0.000000,0.227273,0.500000,0.045455,0.636364,0.000000,0.000000,0.0,0.090909,0.136364,0.045455,0.318182,0.045455,0.045455
2,-WM58wLjtlHlR91xVfM1FQ,0.000000,0.291667,0.416667,0.041667,0.250000,0.083333,0.000000,1.166667,0.083333,0.041667,0.0,0.125000,0.333333,0.000000,0.666667,0.000000,0.125000
3,-qTtg1D3RidRa4cTB-ftwg,0.000000,0.133333,0.200000,0.000000,0.066667,0.000000,0.000000,0.200000,0.000000,0.000000,0.0,0.333333,0.133333,0.000000,0.400000,0.000000,0.000000
4,02H49g16MdRoZKoX6IEoFA,0.136364,0.045455,1.136364,0.000000,0.136364,0.136364,0.000000,0.636364,0.000000,0.090909,0.0,1.318182,0.090909,0.000000,0.272727,0.045455,0.045455


In [75]:
# STEP 39 — MERGE VALUES INTO PERSONAS
# This will allow us to enrich our user personas with insights about the values that are most important to them, 
# which can inform targeted marketing strategies, personalized recommendations, and a deeper understanding of customer segments.

persona_df = persona_df.merge(
    user_values,
    on="user_id",
    how="left"
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,0.722222,0.000000,0.000000,0.0,0.250000,0.027778,0.000000,0.166667,0.027778,0.111111
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.636364,0.000000,0.000000,0.0,0.090909,0.136364,0.045455,0.318182,0.045455,0.045455
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,1.166667,0.083333,0.041667,0.0,0.125000,0.333333,0.000000,0.666667,0.000000,0.125000
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0.200000,0.000000,0.000000,0.0,0.333333,0.133333,0.000000,0.400000,0.000000,0.000000
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,0.636364,0.000000,0.090909,0.0,1.318182,0.090909,0.000000,0.272727,0.045455,0.045455


In [77]:
# STEP 40 — IDENTIFY DOMINANT VALUES
# This will help us understand the key values that define each user persona, allowing us to tailor our marketing strategies and recommendations to align with what matters most to each segment of our customer base.
# We identify the dominant value for each user by finding the value category with the highest average score in their reviews, which can provide insights into their core preferences and priorities.

value_columns = list(value_taxonomy.keys())

persona_df["dominant_value"] = (
    persona_df[value_columns]
    .idxmax(axis=1)
)

persona_df[
    ["user_id", "archetype", "dominant_value"]
].head(11)

,user_id,archetype,dominant_value
0,-EX1hrPRBqNkVavtMllTCA,Harsh Critic,food_quality
1,-M7fUg7FrdGctKr5f_eMUQ,Emotional Storyteller,service_quality
2,-WM58wLjtlHlR91xVfM1FQ,Emotional Storyteller,food_quality
3,-qTtg1D3RidRa4cTB-ftwg,Warm Optimist,positive_exaggeration
4,02H49g16MdRoZKoX6IEoFA,Reactive Reviewer,convenience
5,0650daOKAuufqymyOBe3cA,Warm Optimist,service_quality
6,06Yz-YYYa1U9PN37b6UniA,Emotional Storyteller,food_quality
7,0Tsu6-uhw_w9Z3W0Xlnazg,Harsh Critic,affordability
8,0xJwTzZuWOac7ufeTioeig,Emotional Storyteller,food_quality
9,175DuOm7IPiEy5f1KG9esA,Harsh Critic,food_quality


Personas are no longer generic reviewers. They now contain:

- emotional behavior
- archetypes
- behavioral style
- human priorities
- dominant values

PHASE 2 — TEMPORAL & EMOTIONAL DRIFT MODELING

This is where the system learns that humans evolve.

We will model:

Behavior	              Meaning
- rating drift	          becoming harsher/more generous over time
- sentiment drift	      emotional evolution
- preference drift	      changing tastes
- temporal behavior	      seasonality
- identity evolution	  life-stage transitions

In [ ]:
# STEP 41 — PREPARE TEMPORAL DATA
# We first ensure dates are proper datetime objects.

# Convert review dates to datetime

curated_reviews["date"] = pd.to_datetime(
    curated_reviews["date"]
)

print("Date conversion complete")

Date conversion complete


In [80]:
# STEP 42 — SORT USER REVIEWS CHRONOLOGICALLY


curated_reviews = curated_reviews.sort_values(
    by=["user_id", "date"]
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,...,food_quality,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty
985,6Vf1MkxDPrTcnuKYUldjFw,-EX1hrPRBqNkVavtMllTCA,agK5cXwnBQozM2M-5kLvzw,1,8,2,2,"We had food from this ""restaurant"" delivered t...",2011-12-15 12:02:27,-0.300000,...,2,0,0,0,0,0,0,0,0,0
7114,Aninptga9OOZsmi5gFNAjQ,-EX1hrPRBqNkVavtMllTCA,XnVmNQdmyhdCC90FRY9ZJQ,5,0,0,0,Had our Christmas party here this year. The s...,2011-12-16 23:14:10,0.540179,...,0,0,0,0,0,0,0,1,0,0
107,EiPqGnP5SRapBaOkvZJAug,-EX1hrPRBqNkVavtMllTCA,GBTPC53ZrG1ZBY3DT8Mbcw,4,1,0,1,"Went here on a lark, basically. I had enough ...",2012-02-04 17:10:19,0.298437,...,1,0,0,0,0,0,0,0,0,0
100,pL3LWEwcqaTLaa7c4o860A,-EX1hrPRBqNkVavtMllTCA,-9yzQQ0d_rcOD2CzdTNO_Q,5,1,1,2,"This is one of the ""retro"" McDonald's and it's...",2012-02-07 18:27:15,0.276420,...,0,0,0,0,0,0,0,0,0,0
2331,46Z1SVg6VJygnN3O95cFsw,-EX1hrPRBqNkVavtMllTCA,FEFi0AmjHzgSceeLYW3Glw,2,0,0,0,This location is inconsistent. Sometimes it b...,2012-02-07 18:49:57,-0.150000,...,0,0,0,0,2,0,0,0,0,0


WHY THIS MATTERS

Human evolution only makes sense over time.

Now reviews become behavioral timelines, instead of isolated events.

In [ ]:
# STEP 43 — CREATE TEMPORAL USER HISTORIES
# Here we are creating temporal user histories by aggregating the review dates, star ratings, and sentiment scores for each user into lists.
# This will allow us to analyze how user preferences and sentiments evolve over time, providing insights into user behavior and trends in reviews.

temporal_histories = (
    curated_reviews
    .groupby("user_id")
    .agg({
        "date": list,
        "stars": list,
        "sentiment": list
    })
    .reset_index()
)

temporal_histories.head()

,user_id,date,stars,sentiment
0,-EX1hrPRBqNkVavtMllTCA,"[2011-12-15 12:02:27, 2011-12-16 23:14:10, 201...","[1, 5, 4, 5, 2, 5, 2, 5, 4, 1, 4, 2, 2, 5, 2, ...","[-0.3, 0.5401785714285714, 0.29843749999999997..."
1,-M7fUg7FrdGctKr5f_eMUQ,"[2014-01-03 07:44:42, 2014-01-03 07:56:13, 201...","[2, 4, 4, 1, 5, 4, 5, 5, 5, 5, 5, 4, 4, 5, 5, ...","[0.2, 0.3666666666666667, 0.054166666666666696..."
2,-WM58wLjtlHlR91xVfM1FQ,"[2014-04-21 22:33:53, 2014-08-31 18:31:10, 201...","[1, 5, 5, 5, 4, 5, 3, 5, 5, 5, 5, 5, 5, 5, 3, ...","[-0.13816183816183814, 0.33333333333333337, 0...."
3,-qTtg1D3RidRa4cTB-ftwg,"[2016-11-25 02:54:00, 2016-12-04 18:49:50, 201...","[5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5]","[0.6666666666666666, 0.35, 0.3511904761904762,..."
4,02H49g16MdRoZKoX6IEoFA,"[2014-06-20 07:55:29, 2014-06-20 08:31:14, 201...","[4, 5, 4, 3, 5, 4, 5, 5, 5, 4, 5, 4, 5, 5, 5, ...","[0.014625000000000003, 0.3240165631469979, 0.3..."


WHAT THIS REPRESENTS

Each user now has:

chronological ratings
emotional trajectory
behavioral timeline

This represents a temporal identity.

In [ ]:
# STEP 44 — COMPUTE RATING DRIFT
# We now measure whether users become harsher or softer over time.
# By computing the rating drift, we can identify trends in user behavior, 
# such as whether users tend to become more critical or more lenient in their reviews over time, 
# which can provide insights into changes in user expectations and satisfaction.

def compute_rating_drift(ratings):

    if len(ratings) < 2:
        return 0

    return ratings[-1] - ratings[0]

In [83]:
temporal_histories["rating_drift"] = (
    temporal_histories["stars"]
    .apply(compute_rating_drift)
)

temporal_histories.head()

,user_id,date,stars,sentiment,rating_drift
0,-EX1hrPRBqNkVavtMllTCA,"[2011-12-15 12:02:27, 2011-12-16 23:14:10, 201...","[1, 5, 4, 5, 2, 5, 2, 5, 4, 1, 4, 2, 2, 5, 2, ...","[-0.3, 0.5401785714285714, 0.29843749999999997...",1
1,-M7fUg7FrdGctKr5f_eMUQ,"[2014-01-03 07:44:42, 2014-01-03 07:56:13, 201...","[2, 4, 4, 1, 5, 4, 5, 5, 5, 5, 5, 4, 4, 5, 5, ...","[0.2, 0.3666666666666667, 0.054166666666666696...",3
2,-WM58wLjtlHlR91xVfM1FQ,"[2014-04-21 22:33:53, 2014-08-31 18:31:10, 201...","[1, 5, 5, 5, 4, 5, 3, 5, 5, 5, 5, 5, 5, 5, 3, ...","[-0.13816183816183814, 0.33333333333333337, 0....",2
3,-qTtg1D3RidRa4cTB-ftwg,"[2016-11-25 02:54:00, 2016-12-04 18:49:50, 201...","[5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5]","[0.6666666666666666, 0.35, 0.3511904761904762,...",0
4,02H49g16MdRoZKoX6IEoFA,"[2014-06-20 07:55:29, 2014-06-20 08:31:14, 201...","[4, 5, 4, 3, 5, 4, 5, 5, 5, 4, 5, 4, 5, 5, 5, ...","[0.014625000000000003, 0.3240165631469979, 0.3...",1


In [84]:
# STEP 45 — COMPUTE SENTIMENT DRIFT
# We now measure the evolution of user sentiments over time.
# By computing the sentiment drift, we can identify trends in user emotions, 
# such as whether users tend to become more positive or more negative in their reviews over time, 
# which can provide insights into changes in user experiences and satisfaction.

def compute_sentiment_drift(sentiments):

    if len(sentiments) < 2:
        return 0

    return sentiments[-1] - sentiments[0]

In [85]:
temporal_histories["sentiment_drift"] = (
    temporal_histories["sentiment"]
    .apply(compute_sentiment_drift)
)

temporal_histories.head()

,user_id,date,stars,sentiment,rating_drift,sentiment_drift
0,-EX1hrPRBqNkVavtMllTCA,"[2011-12-15 12:02:27, 2011-12-16 23:14:10, 201...","[1, 5, 4, 5, 2, 5, 2, 5, 4, 1, 4, 2, 2, 5, 2, ...","[-0.3, 0.5401785714285714, 0.29843749999999997...",1,0.365714
1,-M7fUg7FrdGctKr5f_eMUQ,"[2014-01-03 07:44:42, 2014-01-03 07:56:13, 201...","[2, 4, 4, 1, 5, 4, 5, 5, 5, 5, 5, 4, 4, 5, 5, ...","[0.2, 0.3666666666666667, 0.054166666666666696...",3,-0.309773
2,-WM58wLjtlHlR91xVfM1FQ,"[2014-04-21 22:33:53, 2014-08-31 18:31:10, 201...","[1, 5, 5, 5, 4, 5, 3, 5, 5, 5, 5, 5, 5, 5, 3, ...","[-0.13816183816183814, 0.33333333333333337, 0....",2,0.471495
3,-qTtg1D3RidRa4cTB-ftwg,"[2016-11-25 02:54:00, 2016-12-04 18:49:50, 201...","[5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5]","[0.6666666666666666, 0.35, 0.3511904761904762,...",0,-0.263095
4,02H49g16MdRoZKoX6IEoFA,"[2014-06-20 07:55:29, 2014-06-20 08:31:14, 201...","[4, 5, 4, 3, 5, 4, 5, 5, 5, 4, 5, 4, 5, 5, 5, ...","[0.014625000000000003, 0.3240165631469979, 0.3...",1,0.270200


WHAT THIS MEANS

We are now detecting emotional evolution.

Examples:
- increasing positivity
- frustration accumulation
- declining enthusiasm
- emotional fatigue

In [86]:
# STEP 46 — CLASSIFY EMOTIONAL TRAJECTORIES
# We classify users based on their sentiment drift to understand their emotional trajectories over time.

def classify_drift(value):

    if value > 0.5:
        return "becoming_more_positive"

    elif value < -0.5:
        return "becoming_more_negative"

    else:
        return "emotionally_stable"

In [88]:
temporal_histories["emotional_trajectory"] = (
    temporal_histories["sentiment_drift"]
    .apply(classify_drift)
)

temporal_histories.head(10)

,user_id,date,stars,sentiment,rating_drift,sentiment_drift,emotional_trajectory
0,-EX1hrPRBqNkVavtMllTCA,"[2011-12-15 12:02:27, 2011-12-16 23:14:10, 201...","[1, 5, 4, 5, 2, 5, 2, 5, 4, 1, 4, 2, 2, 5, 2, ...","[-0.3, 0.5401785714285714, 0.29843749999999997...",1,0.365714,emotionally_stable
1,-M7fUg7FrdGctKr5f_eMUQ,"[2014-01-03 07:44:42, 2014-01-03 07:56:13, 201...","[2, 4, 4, 1, 5, 4, 5, 5, 5, 5, 5, 4, 4, 5, 5, ...","[0.2, 0.3666666666666667, 0.054166666666666696...",3,-0.309773,emotionally_stable
2,-WM58wLjtlHlR91xVfM1FQ,"[2014-04-21 22:33:53, 2014-08-31 18:31:10, 201...","[1, 5, 5, 5, 4, 5, 3, 5, 5, 5, 5, 5, 5, 5, 3, ...","[-0.13816183816183814, 0.33333333333333337, 0....",2,0.471495,emotionally_stable
3,-qTtg1D3RidRa4cTB-ftwg,"[2016-11-25 02:54:00, 2016-12-04 18:49:50, 201...","[5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5]","[0.6666666666666666, 0.35, 0.3511904761904762,...",0,-0.263095,emotionally_stable
4,02H49g16MdRoZKoX6IEoFA,"[2014-06-20 07:55:29, 2014-06-20 08:31:14, 201...","[4, 5, 4, 3, 5, 4, 5, 5, 5, 4, 5, 4, 5, 5, 5, ...","[0.014625000000000003, 0.3240165631469979, 0.3...",1,0.270200,emotionally_stable
5,0650daOKAuufqymyOBe3cA,"[2014-10-29 14:37:48, 2014-11-11 22:43:58, 201...","[5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5]","[0.18890977443609022, 0.11666666666666665, 0.5...",0,0.498590,emotionally_stable
6,06Yz-YYYa1U9PN37b6UniA,"[2015-11-11 02:59:40, 2015-11-11 03:09:13, 201...","[4, 3, 1, 5, 4, 5, 5, 5, 3, 4, 5, 5, 3, 4, 5, 4]","[0.6769999999999999, 0.19825757575757574, 0.01...",0,-0.512000,becoming_more_negative
7,0Tsu6-uhw_w9Z3W0Xlnazg,"[2015-11-09 16:59:35, 2015-11-09 17:05:24, 201...","[1, 2, 5, 1, 1, 3, 3, 1, 2, 1, 2, 1, 1, 1, 1, 1]","[0.02291666666666667, 0.40208333333333335, 0.5...",0,0.056944,emotionally_stable
8,0xJwTzZuWOac7ufeTioeig,"[2012-10-30 21:59:15, 2013-03-02 18:33:18, 201...","[5, 5, 4, 4, 4, 1, 5, 5, 5, 3, 5, 4, 2, 5, 4, ...","[0.5666666666666667, 0.26583092833092836, 0.25...",-4,-0.829167,becoming_more_negative
9,175DuOm7IPiEy5f1KG9esA,"[2016-05-14 02:41:10, 2016-06-01 01:14:13, 201...","[4, 5, 1, 5, 5, 5, 5, 3, 5, 1, 4, 5, 3, 2, 5, ...","[0.13571428571428573, 0.06441197691197692, 0.0...",0,0.071104,emotionally_stable


In [ ]:
# STEP 47 — SUMMARIZE EMOTIONAL TRAJECTORIES
# This will help us understand the distribution of emotional trajectories among users, 
# providing insights into how user sentiments evolve over time and whether there are common patterns in emotional changes.

stable = (temporal_histories["emotional_trajectory"] == "emotionally_stable").sum()
positive = (temporal_histories["emotional_trajectory"] == "becoming_more_positive").sum()
negative = (temporal_histories["emotional_trajectory"] == "becoming_more_negative").sum()

print("emotionally_stable:", stable)
print("becoming_more_positive:", positive)
print("becoming_more_negative:", negative)

emotionally_stable: 271
becoming_more_positive: 11
becoming_more_negative: 18


In [ ]:
# STEP 48 — INSPECT USERS WITH EMOTIONAL DRIFT
# This will allow us to qualitatively analyze the reviews of users who exhibit significant emotional drift, 
# providing insights into the factors that may contribute to changes in user sentiment over time and how these changes manifest in their reviews.

for trajectory in [
    "becoming_more_positive",
    "becoming_more_negative"
]:

    print("\n" + "="*80)
    print(f"TRAJECTORY: {trajectory}")
    print("="*80)

    # Get sample users
    sampled_users = (
        temporal_histories[
            temporal_histories["emotional_trajectory"]
            == trajectory
        ]
        .sample(3, random_state=42)
    )

    for _, user_row in sampled_users.iterrows():

        user_id = user_row["user_id"]

        print("\n" + "#"*70)
        print(f"USER ID: {user_id}")
        print(f"Trajectory: {trajectory}")
        print(f"Rating Drift: {user_row['rating_drift']}")
        print(f"Sentiment Drift: {user_row['sentiment_drift']}")
        print("#"*70)

        # Get chronological reviews
        user_reviews = (
            curated_reviews[
                curated_reviews["user_id"] == user_id
            ]
            .sort_values("date")
        )

        # FIRST REVIEW
        first_review = user_reviews.iloc[0]

        print("\nFIRST REVIEW")
        print("-"*50)
        print(f"Date: {first_review['date']}")
        print(f"Stars: {first_review['stars']}")
        print(f"Sentiment: {first_review['sentiment']}")

        print(first_review["text"][:500])

        # LAST REVIEW
        last_review = user_reviews.iloc[-1]

        print("\nLAST REVIEW")
        print("-"*50)
        print(f"Date: {last_review['date']}")
        print(f"Stars: {last_review['stars']}")
        print(f"Sentiment: {last_review['sentiment']}")

        print(last_review["text"][:500])

        print("\n\n")


TRAJECTORY: becoming_more_positive

######################################################################
USER ID: OGxTI5DMFWrPWp65_wHWGQ
Trajectory: becoming_more_positive
Rating Drift: 4
Sentiment Drift: 0.7169934640522876
######################################################################

FIRST REVIEW
--------------------------------------------------
Date: 2013-11-10 22:06:47
Stars: 1
Sentiment: 0.060784313725490244
Attention Business Owner,
I first tried your restaurant, Cheba Hut in Tucson, about a month ago and thought it was fun, quirky, quick and tasty. I liked it enough to recommend it to others. Which is why it was stunning to have the complete opposite experience today. Bad customer service takes some of the good taste from food. The general manager, who says he is your brother provided by far the worst CS experience I have had in a while (talking over me, correcting me, etc). Even the employee we w

LAST REVIEW
--------------------------------------------------
Date:

In [ ]:
# STEP 49 — MERGE TEMPORAL SIGNALS INTO PERSONAS
# This will allow us to enrich our user personas with insights about how their sentiments and ratings evolve over time,

persona_df = persona_df.merge(
    temporal_histories[
        [
            "user_id",
            "rating_drift",
            "sentiment_drift",
            "emotional_trajectory"
        ]
    ],
    on="user_id",
    how="left"
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty,dominant_value,rating_drift,sentiment_drift,emotional_trajectory
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,0.250000,0.027778,0.000000,0.166667,0.027778,0.111111,food_quality,1,0.365714,emotionally_stable
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.090909,0.136364,0.045455,0.318182,0.045455,0.045455,service_quality,3,-0.309773,emotionally_stable
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.125000,0.333333,0.000000,0.666667,0.000000,0.125000,food_quality,2,0.471495,emotionally_stable
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0.333333,0.133333,0.000000,0.400000,0.000000,0.000000,positive_exaggeration,0,-0.263095,emotionally_stable
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,1.318182,0.090909,0.000000,0.272727,0.045455,0.045455,convenience,1,0.270200,emotionally_stable


WHAT WE NOW POSSESS: A real cognitive persona engine.

The personas now include:

Dimension	         Meaning
archetype	         behavioral identity
dominant_value	     priorities
sentiment	         emotionality
drift	             evolution
trajectory	         temporal behavior




PHASE 3: PREFERENCE DRIFT MODELLING

This models changing interests over time.

Example:

2019:
cheap fast food

2023:
upscale aesthetic dining

That implies income change, identity change, maturity, lifestyle evolution etc.

We will detect:

- changing restaurant categories
- changing review topics
- evolving priorities
- shifting preferences

In [94]:
# STEP 50 — EXTRACT BUSINESS CATEGORIES
# 

business_subset = business_df[
    ["business_id", "categories"]
].copy()

business_subset.head()

,business_id,categories
0,Pns2l4eNsfO8kk83dixA6A,"Doctors, Traditional Chinese Medicine, Naturop..."
1,mpf3x-BjTdTEA3yCZrAYPw,"Shipping Centers, Local Services, Notaries, Ma..."
2,tUFrWirKiKi_TAnsVWINQQ,"Department Stores, Shopping, Fashion, Home & G..."
3,MTSW4McQd7CbVtyjqoe9mw,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
4,mWMc6_wTdE0EUBKIGXDVfA,"Brewpubs, Breweries, Food"


In [95]:
# STEP 51 — MERGE BUSINESS CATEGORIES INTO REVIEWS

curated_reviews = curated_reviews.merge(
    business_subset,
    on="business_id",
    how="left"
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,...,electronics,fashion,books_media,convenience,social_experience,luxury,positive_exaggeration,negative_exaggeration,uncertainty,categories
0,6Vf1MkxDPrTcnuKYUldjFw,-EX1hrPRBqNkVavtMllTCA,agK5cXwnBQozM2M-5kLvzw,1,8,2,2,"We had food from this ""restaurant"" delivered t...",2011-12-15 12:02:27,-0.300000,...,0,0,0,0,0,0,0,0,0,"Italian, American (Traditional), Restaurants, ..."
1,Aninptga9OOZsmi5gFNAjQ,-EX1hrPRBqNkVavtMllTCA,XnVmNQdmyhdCC90FRY9ZJQ,5,0,0,0,Had our Christmas party here this year. The s...,2011-12-16 23:14:10,0.540179,...,0,0,0,0,0,0,1,0,0,"Restaurants, Steakhouses, Seafood, Nightlife, ..."
2,EiPqGnP5SRapBaOkvZJAug,-EX1hrPRBqNkVavtMllTCA,GBTPC53ZrG1ZBY3DT8Mbcw,4,1,0,1,"Went here on a lark, basically. I had enough ...",2012-02-04 17:10:19,0.298437,...,0,0,0,0,0,0,0,0,0,"German, Restaurants, Seafood, Cocktail Bars, F..."
3,pL3LWEwcqaTLaa7c4o860A,-EX1hrPRBqNkVavtMllTCA,-9yzQQ0d_rcOD2CzdTNO_Q,5,1,1,2,"This is one of the ""retro"" McDonald's and it's...",2012-02-07 18:27:15,0.276420,...,0,0,0,0,0,0,0,0,0,"Fast Food, Restaurants, Coffee & Tea, Food, Bu..."
4,46Z1SVg6VJygnN3O95cFsw,-EX1hrPRBqNkVavtMllTCA,FEFi0AmjHzgSceeLYW3Glw,2,0,0,0,This location is inconsistent. Sometimes it b...,2012-02-07 18:49:57,-0.150000,...,0,0,0,2,0,0,0,0,0,"Food, Burgers, Coffee & Tea, Fast Food, Restau..."


In [96]:
# STEP 52 — CLEAN CATEGORY TEXT

curated_reviews["categories"] = (
    curated_reviews["categories"]
    .fillna("")
)

In [97]:
# STEP 53 — CREATE SIMPLE CUISINE TAGS
# We now infer preference domains.


cuisine_keywords = [
    "Mexican",
    "Italian",
    "Chinese",
    "Japanese",
    "Thai",
    "Indian",
    "American",
    "Mediterranean",
    "Korean",
    "French",
    "Pizza",
    "Seafood",
    "Burgers",
    "Cafe",
    "Bars"
]

In [ ]:
# STEP 54 — EXTRACT CUISINE PREFERENCES

def extract_cuisines(category_text):

    found = []

    for cuisine in cuisine_keywords:

        if cuisine.lower() in category_text.lower():
            found.append(cuisine)

    return found

In [99]:
curated_reviews["cuisines"] = (
    curated_reviews["categories"]
    .apply(extract_cuisines)
)

curated_reviews[
    ["categories", "cuisines"]
].head()

,categories,cuisines
0,"Italian, American (Traditional), Restaurants, ...","[Italian, American]"
1,"Restaurants, Steakhouses, Seafood, Nightlife, ...","[American, Seafood, Bars]"
2,"German, Restaurants, Seafood, Cocktail Bars, F...","[American, French, Seafood, Bars]"
3,"Fast Food, Restaurants, Coffee & Tea, Food, Bu...",[Burgers]
4,"Food, Burgers, Coffee & Tea, Fast Food, Restau...",[Burgers]


In [100]:
# STEP 55 — SPLIT EARLY VS RECENT BEHAVIOR
# We compare past self vs current self.

def split_temporal_preferences(user_df):

    user_df = user_df.sort_values("date")

    midpoint = len(user_df) // 2

    early = user_df.iloc[:midpoint]
    recent = user_df.iloc[midpoint:]

    return early, recent

In [101]:
# STEP 56: DEYECT PREFERENCE DRIFT

from collections import Counter

preference_drift_results = []

for user_id, user_df in curated_reviews.groupby("user_id"):

    if len(user_df) < 6:
        continue

    early, recent = split_temporal_preferences(user_df)

    early_cuisines = [
        cuisine
        for sublist in early["cuisines"]
        for cuisine in sublist
    ]

    recent_cuisines = [
        cuisine
        for sublist in recent["cuisines"]
        for cuisine in sublist
    ]

    early_top = Counter(early_cuisines).most_common(3)
    recent_top = Counter(recent_cuisines).most_common(3)

    preference_drift_results.append({
        "user_id": user_id,
        "early_preferences": early_top,
        "recent_preferences": recent_top
    })

In [102]:
# STEP 57 — CREATE PREFERENCE DRIFT DATAFRAME

preference_drift_df = pd.DataFrame(
    preference_drift_results
)

preference_drift_df.head()

,user_id,early_preferences,recent_preferences
0,-EX1hrPRBqNkVavtMllTCA,"[(American, 7), (Burgers, 7), (Bars, 4)]","[(Burgers, 4), (American, 3), (Bars, 3)]"
1,-M7fUg7FrdGctKr5f_eMUQ,"[(Mexican, 5), (American, 4), (Seafood, 4)]","[(Bars, 7), (American, 5), (Mediterranean, 2)]"
2,-WM58wLjtlHlR91xVfM1FQ,"[(American, 3), (Mexican, 2), (Burgers, 2)]","[(Cafe, 2), (Mediterranean, 2), (Seafood, 2)]"
3,-qTtg1D3RidRa4cTB-ftwg,"[(American, 4), (Seafood, 3), (Cafe, 1)]","[(American, 7), (Bars, 6), (Cafe, 2)]"
4,02H49g16MdRoZKoX6IEoFA,"[(American, 4), (Bars, 2), (Pizza, 2)]","[(Mexican, 3), (Burgers, 2), (American, 2)]"


In [ ]:
# Inspect a few users
preference_drift_df.sample(15)

,user_id,early_preferences,recent_preferences
45,Bezh6tDmCrL0-SnsV6CneA,"[(American, 6), (Bars, 4), (French, 3)]","[(American, 5), (Seafood, 4), (Bars, 3)]"
0,-EX1hrPRBqNkVavtMllTCA,"[(American, 7), (Burgers, 7), (Bars, 4)]","[(Burgers, 4), (American, 3), (Bars, 3)]"
281,xIgBho-drdd-sXcPebxtrg,"[(Bars, 4), (Japanese, 1), (Italian, 1)]","[(American, 3), (Cafe, 2), (Seafood, 1)]"
244,ppl2TnSwu7SIt2r94-LoDA,"[(Bars, 9), (American, 7), (Mexican, 3)]","[(Bars, 7), (American, 5), (Italian, 4)]"
21,3kvIOBG06_rikfpk-EHIlQ,"[(American, 6), (Bars, 6), (Burgers, 2)]","[(American, 6), (Bars, 6), (Burgers, 1)]"
67,FTaNwFkvoHOX8DkGya7NKQ,"[(American, 3), (Bars, 2), (Burgers, 2)]","[(Burgers, 2), (Chinese, 1), (Mediterranean, 1)]"
235,oY9BWzpsvAJojnahGMtDIw,"[(Seafood, 4), (American, 4), (Bars, 3)]","[(Bars, 6), (American, 3), (Seafood, 2)]"
68,FnQvnUkRopiaz9lcmz0Jew,"[(Bars, 4), (Japanese, 2), (Italian, 1)]","[(Mediterranean, 2), (Bars, 2), (Mexican, 1)]"
78,HvHR1rLqRvEzXYoyY6cxsg,"[(Bars, 7), (American, 6), (Italian, 3)]","[(Bars, 8), (American, 6), (Pizza, 3)]"
9,175DuOm7IPiEy5f1KG9esA,"[(Bars, 8), (American, 7), (Pizza, 2)]","[(American, 9), (Seafood, 6), (Bars, 6)]"


Possible interpretation for preference drift of these users:

maturing tastes
luxury orientation
lifestyle evolution

This shows human evolution modeling.